# LEACE Concept Erasure Recipe: delete grammatical number from GPT-2 small

LEAst-squares Concept Erasure (Belrose et al. 2023, "LEACE: Perfect linear concept erasure in closed form", NeurIPS 2023, arXiv:2306.03819) is the smallest affine change to a representation after which no linear classifier can recover a concept. It is closed form: whiten, project out the span of the concept's cross-covariance, unwhiten. This notebook fits LEACE for subject number (singular versus plural) on the output of a middle block of GPT-2 small, applies it at capture time with one `tl.when` rule, and measures two things: a linear probe for number falls to chance, and the model's subject-verb agreement degrades, while a control that erases a random direction of the same rank through the builtin `subspace_ablate` changes neither.

It downloads GPT-2 small (about 550 MB) at a pinned revision and runs on a laptop CPU in about a minute.

In [ ]:
from __future__ import annotations

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

import torchlens as tl

MODEL_ID = "openai-community/gpt2"
REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"  # pragma: allowlist secret
torch.manual_seed(0)
model = GPT2LMHeadModel.from_pretrained(
    MODEL_ID, revision=REVISION, attn_implementation="eager", use_cache=False
).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
SITE = "transformer.h.6"  # the block whose output carries the concept we erase
IS, ARE = tokenizer.encode(" is")[0], tokenizer.encode(" are")[0]

## Sentences whose next word must agree with the subject

Sixteen nouns in four frames, singular and plural, each ending where the verb goes. Every noun is one GPT-2 token, so all sentences of one frame have the same length and batch without padding.

In [ ]:
NOUNS = [
    ("cat", "cats"),
    ("dog", "dogs"),
    ("car", "cars"),
    ("book", "books"),
    ("teacher", "teachers"),
    ("bird", "birds"),
    ("house", "houses"),
    ("tree", "trees"),
    ("key", "keys"),
    ("child", "children"),
    ("boy", "boys"),
    ("girl", "girls"),
    ("door", "doors"),
    ("river", "rivers"),
    ("apple", "apples"),
    ("horse", "horses"),
]
FRAMES = [
    "The {} near the window",
    "The {} in the garden",
    "The {} by the river",
    "I think the {} on the table",
]
batches, plural = [], []
for frame in FRAMES:
    texts = [
        frame.format(form) for singular, plural_form in NOUNS for form in (singular, plural_form)
    ]
    ids = [tokenizer.encode(text) for text in texts]
    assert len({len(row) for row in ids}) == 1, "one frame, one length"
    batches.append(torch.tensor(ids))
    plural += [0.0, 1.0] * len(NOUNS)
plural = torch.tensor(plural)
print(
    len(plural),
    "sentences, e.g.",
    [tokenizer.decode(batches[0][0]), tokenizer.decode(batches[0][1])],
)


def run(intervene=None):
    """Trace every frame; return the block outputs per position and the is/are logits at the end."""
    hidden, verb_logits = [], []
    for ids in batches:
        trace = tl.trace(model, ids, intervene=intervene)
        hidden.append(trace[SITE].out.detach())
        verb_logits.append(trace.modules["lm_head"].out[:, -1, [IS, ARE]].detach())
    return hidden, torch.cat(verb_logits)


def agreement(verb_logits: torch.Tensor) -> float:
    """Fraction of sentences whose preferred verb form agrees with the subject."""
    return ((verb_logits[:, 1] > verb_logits[:, 0]).float() == plural).float().mean().item()


hidden, verb_logits = run()
print(f"agreement accuracy before erasure: {agreement(verb_logits):.2f}")

## Fit LEACE in closed form

With `x` the block's output at every position (labelled by its sentence's number) and `W` the ZCA whitening of `x`, LEACE's projection is `P = W^+ proj(W Sigma_xz) W` and the erased value is `x - P (x - mean)` (Belrose et al. 2023, Theorem 4.1).

In [ ]:
def fit_leace(x: torch.Tensor, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return ``(mean, P)`` such that ``x - P (x - mean)`` erases ``z`` linearly."""
    x, z = x.double(), z.double()[:, None]
    mean = x.mean(0)
    x_centered, z_centered = x - mean, z - z.mean(0)
    sigma_xx = x_centered.T @ x_centered / len(x)
    sigma_xz = x_centered.T @ z_centered / len(x)
    evals, evecs = torch.linalg.eigh(sigma_xx)
    keep = evals > evals.max() * 1e-8
    vecs, vals = evecs[:, keep], evals[keep]
    whiten = (vecs / vals.sqrt()) @ vecs.T  # Sigma_xx^(-1/2)
    unwhiten = (vecs * vals.sqrt()) @ vecs.T  # Sigma_xx^(1/2)
    basis, _ = torch.linalg.qr(whiten @ sigma_xz)  # orthonormal span of W Sigma_xz
    return mean.float(), (unwhiten @ basis @ basis.T @ whiten).float()


positions = torch.cat([h.reshape(-1, h.shape[-1]) for h in hidden])
labels = torch.cat(
    [
        plural[i * len(h) : (i + 1) * len(h)].repeat_interleave(h.shape[1])
        for i, h in enumerate(hidden)
    ]
)
mean, projection = fit_leace(positions, labels)
print(
    "fitted on",
    tuple(positions.shape),
    "vectors; erasure rank",
    int(torch.linalg.matrix_rank(projection)),
)

## Does the concept survive a linear probe?

A logistic-regression probe on the last position before erasure, and on the same states after the erasure map. Held-out nouns test generalisation; on the fitting data LEACE guarantees the probe cannot beat the constant classifier.

In [ ]:
def probe_accuracy(
    features: torch.Tensor, targets: torch.Tensor, train: torch.Tensor
) -> tuple[float, float]:
    """Train a logistic probe on ``train`` rows; return (train, held-out) accuracy."""
    feats = (features - features[train].mean(0)) / (features[train].std(0) + 1e-6)
    weight = torch.zeros(feats.shape[1], requires_grad=True)
    bias = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([weight, bias], max_iter=200, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            feats[train] @ weight + bias, targets[train]
        )
        loss = loss + 1e-2 * weight.square().sum()
        loss.backward()
        return loss

    optimizer.step(closure)
    predicted = ((feats @ weight + bias) > 0).float()
    correct = predicted == targets
    return correct[train].float().mean().item(), correct[~train].float().mean().item()


last = torch.cat([h[:, -1] for h in hidden])
train = torch.tensor(
    [noun < 12 for _frame in FRAMES for noun in range(len(NOUNS)) for _form in range(2)]
)
before = probe_accuracy(last, plural, train)
after = probe_accuracy(last - (last - mean) @ projection.T, plural, train)
print(
    f"probe accuracy (train, held-out): before {before[0]:.2f}, {before[1]:.2f}; after LEACE {after[0]:.2f}, {after[1]:.2f}"
)
assert before[1] > 0.85, "number is linearly readable at this block"
assert after[0] < 0.65, "LEACE leaves no linearly readable number"

## Apply the erasure inside the forward pass

One `tl.when` rule applies the map to the block's output at every position of every sentence while the model runs. The control erases one random direction with the builtin `subspace_ablate(mode="mean")`, the same rank and the same mean-matching, so any difference comes from the direction LEACE chose.

In [ ]:
def erase(activation: torch.Tensor, *, ctx) -> torch.Tensor:
    """LEACE: ``x - P (x - mean)``."""
    return activation - (activation - mean) @ projection.T


random_direction = torch.randn(1, mean.shape[0], generator=torch.Generator().manual_seed(1))
control = tl.intervention.subspace_ablate(random_direction, mode="mean", mean=mean, dim=-1)

_, erased_logits = run(intervene=tl.when(tl.module(SITE), erase))
_, control_logits = run(intervene=tl.when(tl.module(SITE), control))
print(
    f"agreement accuracy: before {agreement(verb_logits):.2f}, LEACE {agreement(erased_logits):.2f}, random-direction control {agreement(control_logits):.2f}"
)
print(
    f"mean |is-are| logit shift: LEACE {(erased_logits - verb_logits).abs().mean():.2f}, control {(control_logits - verb_logits).abs().mean():.2f}"
)
assert agreement(erased_logits) < agreement(verb_logits), "erasing number should hurt agreement"
assert abs(agreement(control_logits) - agreement(verb_logits)) <= 0.1, (
    "a random direction should not"
)